# RealModel Exercise – CMIP6 future temperature projections

### Background: What is CMIP6?
In this short exercise, you will work with real Earth system model (ESM) data. The different ESMs used here all participate in the Coupled Model Intercomparison Project - Phase 6 (CMIP6), and are used by the Intergovernmental Panel on Climate Change (IPCC) to inform statements about the physical state of the climate and future global warming.  

CMIP6 is a project which coordinates experiments with ESMs across modelling centers. In order to participate, modelling centers (a modelling center typically operates one ESM) have to conduct specific, controlled experiments with their ESM. The setup of these experiments is identical for all participating ESMs. This allows scientists to analyze data from different ESMs which is comparable, because the experimental setup was identical. 

Such experiments could be, for example, simulating the historical climate from 1850 until 2014. All modelling centers then run their ESM with pre-defined boundary conditions as external forcing (for example, precise solar radiation, volcanic eruptions, greenhouse-gas emissions, etc.). The forcing data is provided by CMIP6 and is, of course, identical for all participating ESMs. 

### Your Task
Here, we provide you with annual global surface air temperature (GSAT) data from 20 ESMs participating in CMIP6. The data consist of a minimum of 5 ensemble members per model. The ensemble members are a historical experiment spanning 1850-2014, smoothly continued by a scenario simulation from 2015 until 2100, under the so-called SSP2-4.5 scenario. 

Your ultimate task will be to make a so-called "emergent constraint" based on this data, which tells policy makers how much warming to expect by the end of the century, if we follow a SSP2-4.5 scenario. 

Don't worry if the concept of an emergent constraint is not clear to you at this point. By following the step-by-step guide in the Jupyter Notebook, you should be able to grasp the concepts and execute the exercise. 

This exercise is strongly based on existing work done with emergent constraints, in particular: 
- Tokarska KB, Stolpe MB, Sippel S, Fischer EM, Smith CJ, Lehner F, Knutti R. Past warming trend constrains future warming in CMIP6 models. Sci Adv. 2020 Mar 18;6(12):eaaz9549. doi: 10.1126/sciadv.aaz9549. PMID: 32206725; PMCID: PMC7080456.

Whenever you feel stuck and are unsure how to answer some of the questions, especially those requiring interpretation of your results, we recommend reading the Tokarska et al. (2020) study. 

### Have fun!

----------------------------------------------------------------------------------

# Start

We provide you with a pd.DataFrame object (https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) containing the relevant GSAT data for this exercise. Execute all existing code cells and fill out code cells marked with #TODO. Answer text questions in raw, also marked with TODO. We recommend you to work with pandas dataframes throughout the exercise, as this allows easy storage of metadata (such as Model, Year, etc.) together with the actual data we care about (GSAT). 


In [ ]:
import pandas as pd
import numpy as np
import os
import xarray as xr
import pickle
from tqdm import tqdm
import seaborn as sns
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib as mpl

In [ ]:
# Set the path to the data directory
datapath = 'data/'

# Load dataframe with cmip6 sufrace temperature data
with open(f'{datapath}cmip6_surface_temperature_5mems.pkl', 'rb') as file: 
    df_temp = pickle.load(file)


### 1. Temperature change in ESMs
First, familiarize yourselves with the dataframe structure (Hint: pd.DataFrame.head(n) shows the top n rows of a pandas DataFrame). 
 

In [ ]:
display(df_temp.head())
display(df_temp.shape)

The dataframe contains a column containing the GSAT, a column indicating which ESM this is, another column with a string indicating the precise ensemble member, and a column with the simulation year. You can also print out all unique elements of a column like this:

In [ ]:
# See what ESMs are present in the dataset
pd.Series(df_temp.Model.unique())

In Figure 1 below, you can see the the model output for 20 ESMs for the historical period until 2014 and the future SSP2-4.5 projection until 2100. The thick colored lines show the ensemble mean (average across all runs of an ESM), the shading shows the minimum and maximum of the ensemble.

(NOTE: The function sns.lineplot() automatically displays the mean across a group (thick line). The grouping variable can be specified with the argument 'hue' in the function. See https://seaborn.pydata.org/generated/seaborn.lineplot.html )

In [ ]:
# Visualize data
ax = sns.lineplot(df_temp, x='Year', y='Temperature', hue='Model', errorbar=('pi',100))
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
ax.set_ylabel('Global Mean Surface Temperature $(\degree C)$')
plt.suptitle('Figure 1',
             x=0.1, y=0.02, fontsize=12, fontweight='bold')

plt.show()

__1.a) What are the three hottest models by the end of the 21st century?__

Hint 1: you can use the function pd.DataFrame.groupby() to group your dataframe by a specific variable and compute statistics like the mean. The function pd.Series.sort_values() allows you to sort a series in ascending or descending order.

Hint 2: Think about what a good definition of "end of century" is and choose your own. Is looking at the year 2100 enough?

In [ ]:
#TODO: your code here


__1.b) We now want to look at anomalies with respect to the reference period 1981-2014, instead of the raw data. Compute the anomalies with respect to this period and reproduce the plot from Figure 1 by replacing the raw data with the anomalies.__

Hint: Start by computing the reference period temperatures for each model and creating a column which stores them. Then subtract this from the raw temperature to obtain anomalies.

In [ ]:
#TODO: your code here


__1.c) What are the three hottest models at the end of the 21st century when we look at GSAT anomalies with respect to 1981–2014 instead of the raw data?__ 

In [ ]:
# TODO: your code here


__1.d) Consider the third hottest model in the raw GSAT ranking from 1.a. How did its ranking change when you transformed the data to anomalies?__



__1.e) Explain why it makes more sense to look at anomalies with respect to a reference period rather than absolute warming when comparing temperature projections by different ESMs.__


### 2. Relationship between historical and future warming


__From now on we will only work with the anomalies you computed in 1.c).__

__2.a) Each ESM ensemble consists of several members, each member being a possible realization of the future. Computing the ensemble mean across all members of an ESM provides a single projection per ESM. What type of uncertainty is reduced by taking the average across ensemble members?__

__2.b) How does the variance scale with the number of members considered? Let's take a brief detour and test this__
- (i) Select the model containing the most members (or at least more than 30)
- (ii) Calculate the anomalies of each member of the selected model with respect to the mean of the ensemble member in the 1981-2014 period

Hint: It might be useful to restructure the data into a matrix, something like rows=Year, columns=run, values=Anomaly

In [ ]:
# TODO: your code here


__2.c) Use a Monte Carlo simulation approach__
- For each ensemble size  
   \( N = 1, 2, ..., M \) where M is the total number of ensemble members:

   - Randomly select **\(N\) ensemble members** (with or without replacement, your choice).
   - Compute the **ensemble mean time series** across the selected members.
   - Calculate the **sample variance over time** of this ensemble mean.

- Repeat the experiment 200 times for each ensemble size \(N\).

- For each ensemble size, compute the **median variance** across all repetitions.

- Plot the **median variance of the ensemble-mean time series as a function of ensemble size \(N\)** and compare it with the **theoretical \(1/N\) reference curve**, scaled to match the variance at \(N=1\). This allows you to assess whether the reduction in variance with increasing ensemble size follows the expected \(1/N\) behavior.


In [ ]:
# TODO: your code here


__2.d) Does the result look correct? If it does not, why? Can you think of any component we missed that can create a difference compared to the theoretical trend? Try to identify the mistake and replot the result!__

Hint: If you are stuck, continue with 2.e) and revisit this exercise afterward.



In [ ]:
# TODO: your code here


__2.e) Compute the ensemble mean GSAT for all ESMs and plot its evolution from 1850 to 2100.__

In [ ]:
#TODO: Your code here


__From now on, we only work with the ensemble mean time series as computed in 2.e).__

__2.f) Compute the CMIP6 multi-model mean warming for 2071-2100, as projected from the entire CMIP6 ensemble.__

In [ ]:
#TODO: Your code here


__2.g) Calculate the linear trend in GSAT for the historical period 1981-2014 for each ESM. For each ESM calculate also the GSAT at the end-of-century, which for this exercise we define as 2071-2100. Plot these in a scatterplot of historical warming (horizontal axis) versus future warming (vertical axis), where each dot represents one ESM.__

Hint: you can use pd.DataFrame.query() to select parts of a dataframe based on conditions for the values in certain columns. 

In [ ]:
#TODO: Your code here


__2.h) Add a line of best fit (linear regression line through the point cloud) to your plot and compute the Pearson correlation between the historical trends and future warming.__

Hint: You can use sns.regplot() for directly plotting the best linear fit through a point cloud without having to calculate a linear regression. 

In [ ]:
#TODO: Your code here


__2.i) What do you observe? What is a possible reason for this relationship?__

### 3. Emergent constraint using observational data
We now want to compare observed historical warming to modelled historical warming in order to create an emergent constraint for end-of-century warming. Here we provide you with a reanalysis dataset, ERA5. The dataset contains annual reanalysis GSAT data from 1940 to 2024. 


__Citation for the dataset:__

ERA5 data can be accessed via https://doi.org/10.24381/cds.f17050d7. 

- Hersbach, H., Bell, B., Berrisford, P., Hirahara, S., Horanyi, A., Munoz-Sabater, J., . . . Thepaut, J.-N. (2020). The ERA5 global reanalysis. Quarterly Journal
of the Royal Meteorological Society, 146 (730), 1999–2049. Retrieved 2024-12-16, from https://onlinelibrary.wiley.com/doi/abs/10.1002/qj.3803 ( eprint: https://onlinelibrary.wiley.com/doi/pdf/10.1002/qj.3803) doi:10.1002/qj.3803

__3.a) Load the observational dataset and inspect it (code below). Plot the observational data together with the CMIP6 models in a sensible way.__



In [ ]:
# Load dataframe with ERA5 data
with open(f'{datapath}era5_GSAT.pkl', 'rb') as file: 
    df_era5 = pickle.load(file)

display(df_era5.head())

# TODO: Your code here


__3.b) Just by looking at the plot from 3.a), are there any models you would trust more or less than others, and why?__

__3.c) Plot the observations a vertical line in your scatter plot from 2.a).__


In [ ]:
#TODO: Your code here


__3.d) Use the relationship between historical and future warming to provide a point estimate of a "likely" future warming based on the concept of an emergent constraint.__

In [ ]:
#TODO: Your code here


__3.e) Compare your estimate of end-of-century warming based on the unconstrained multi-model mean (2.c) and the constrained estimate from 3.d).__
    
- (i) Which estimate would you trust more and why? (max. 2 sentences)
- (ii) What factors could bias the constrained estimate in 3.d)? (max. 2 sentences)

__3.f) Repeat the analysis in Section 2 and 3 but this time calculating the historical warming trend in models and observations for the last 35 years, so 1990-2024:__
- (i) Reproduce the scatterplot from 3.c) and compute the Pearson correlation between historical (1990-2024) and future warming as in 2.d).
- (ii) Produce a new point estimate of future warming like in 3.d).

   __How have the results changed? Does your trust in the new point estimate change? Answer in max. 3 sentences.__ 


In [ ]:
# TODO: Your code here


__3.g) Can you think of a reason why the point estimate of future warming based on the emergent constraint changed when changing the analysis period from 1981-2014 in 3.d) to 1990-2024 in 3.f)?__

__3.h) If you had to provide a "likely" range instead of just a point estimate of end-of-century warming, what kind of uncertainties would you need to take into account?__